In [3]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
import numpy as np
print(np.__version__)


2.0.2


In [5]:
!bash /content/drive/MyDrive/V2_drift/texturemodel/code/run/extract_texture_feats.sh


Streaming output truncated to the last 5000 lines.
Getting features for pRF [x, y, sigma]:
[np.float32(0.22715162), np.float32(0.09408929), np.float32(0.27144176)]
Computing higher order correlations...
time elapsed = 6.59792
Final size of features concatenated is [800 x 641]
Feature types included are:
['pixel_stats', 'mean_magnitudes', 'mean_realparts', 'marginal_stats_lowpass_recons', 'variance_highpass_resid', 'magnitude_feature_autocorrs', 'lowpass_recon_autocorrs', 'highpass_resid_autocorrs', 'magnitude_within_scale_crosscorrs', 'real_within_scale_crosscorrs', 'magnitude_across_scale_crosscorrs', 'real_imag_across_scale_crosscorrs', 'real_spatshift_within_scale_crosscorrs', 'real_spatshift_across_scale_crosscorrs']
Final size of features concatenated is [800 x 641]
Model 901, min/max of features in batch: [-4.379860877990723, 7971.93798828125]
Getting features for pRF [x, y, sigma]:
[np.float32(0.17385432), np.float32(0.17385432), np.float32(0.27144176)]
Computing higher order co

In [ ]:
drive.flush_and_unmount()

In [ ]:
import sys
sys.path.append("/content/drive/MyDrive/V2_drift/texturemodel/code")

import numpy as np
from utils import nsd_utils

x = (np.random.rand(2,3,256,256)*255).astype(np.uint8)
bw = nsd_utils.image_uncolorize_fn(x)
print(bw.shape, bw.dtype, bw.min(), bw.max())



Dask
BW shape: (2, 1, 240, 240)
(2, 1, 240, 240) float32 0.0036650982 0.9893522


check debug file

In [ ]:
import h5py, numpy as np, os

path = "/content/drive/MyDrive/V2_drift/features/pyramid_texture/DEBUG/S1_features_each_prf_4ori_4sf_grid5_npix256.h5py"
with h5py.File(path, "r") as f:
    X = f["features"]

    # Look only at computed region: first 200 images, first pRF (0)
    A = X[0:200, :, 0]  # (200, 641)

    # For each feature dimension, count how many images are exactly zero
    zero_counts = np.sum(A == 0, axis=0)  # (641,)

    # Report worst offenders
    top = np.argsort(-zero_counts)[:10]
    print("Top zero-count features (index: zero_count):")
    for idx in top:
        print(idx, int(zero_counts[idx]))

    print("Features that are zero for ALL 200 images:", int(np.sum(zero_counts == 200)))


Top zero-count features (index: zero_count):
4 74
639 0
638 0
637 0
636 0
635 0
634 0
633 0
632 0
631 0
Features that are zero for ALL 200 images: 0


In [ ]:
import h5py
import os
import numpy as np

fn = "/content/drive/MyDrive/V2_drift/features/pyramid_texture/S5_features_each_prf_4ori_4sf_grid5_npix256.h5py"

# 1) File-level checks
print("Exists:", os.path.exists(fn))
print("File size (GB):", os.path.getsize(fn) / 1e9)

# 2) Open HDF5 safely
with h5py.File(fn, "r") as f:
    print("\nDatasets in file:", list(f.keys()))

    feats = f["features"]

    # 3) Shape & dtype
    print("\nFeature dataset info:")
    print("  shape:", feats.shape)
    print("  dtype:", feats.dtype)

    # Expected: (10000, n_features, n_prfs)
    n_images, n_features, n_prfs = feats.shape
    print(f"\nParsed dimensions:")
    print("  n_images :", n_images)
    print("  n_features:", n_features)
    print("  n_prfs    :", n_prfs)

    # 4) Sanity check a few slices (no full read)
    sample_inds = [0, n_images // 2, n_images - 1]
    for idx in sample_inds:
        x = feats[idx, :, :]
        print(f"\nImage {idx}:")
        print("  min:", float(np.min(x)))
        print("  max:", float(np.max(x)))
        print("  any NaNs:", np.isnan(x).any())
